# 原 Stage1 的 NVDA 指引实现状态输入预检

按 `source_preflight_plan.json` 只读既有Qlib Recorder，验证原Stage1 2011–2022训练日历和2023+完整账户日历的来源身份、09:29决策时点及已知状态。0拟合、0账户、0市场收益读取。

In [1]:
from pathlib import Path
import hashlib,json
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.nvda_own_guidance_realization import align_realizations,FEATURE,FLAGS
root=Path.cwd();card=root/'research/stage1_nvda_guidance_receiver_20261022'
plan=json.loads((card/'source_preflight_plan.json').read_text())
objects={};summaries={}
for name,source in plan['sources'].items():
    path=root/source['card'];s=json.loads((path/'summary.json').read_text())
    assert s['recorder_id']==source['recorder_id']
    manifest=json.loads((path/'verification_manifest.json').read_text())
    hits=[x for x in manifest['entries'] if x['path'].endswith('/artifacts/'+source['artifact'])]
    assert len(hits)==1 and hits[0]['sha256']==source['sha256']
    assert hashlib.sha256((path/hits[0]['path']).read_bytes()).hexdigest()==source['sha256']
    summaries[name]=s
qlib.init(provider_uri=summaries['training_timing']['provider_uri'],region='us')
for name,source in plan['sources'].items():
    s=summaries[name]
    with R.uri_context(s['recorder_uri']):
        recorder=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
        objects[name]=recorder.load_object(source['artifact'])
print('Three source SHA identities and Recorder reads passed.')

In [2]:
timing=objects['training_timing'];frame=objects['evaluation_calendar'];pairs=objects['guidance_pairs']
train_timing=timing.loc[timing.index<pd.Timestamp('2023-01-01'),'decision_at']
assert len(train_timing)==3020 and len(frame)==944
eval_index=pd.DatetimeIndex(frame.signal_date)
eval_trade=pd.to_datetime(frame.trade_date).dt.tz_localize('America/New_York')
eval_timing=pd.Series(eval_trade.array+pd.Timedelta(hours=9,minutes=29),index=eval_index)
assert eval_timing.dt.hour.eq(9).all() and eval_timing.dt.minute.eq(29).all()
assert (eval_timing.reset_index(drop=True)<pd.to_datetime(frame.decision_at).dt.tz_localize('America/New_York')).all()
train_features,train_trace,events=align_realizations(pairs,train_timing)
eval_features,eval_trace,_=align_realizations(pairs,eval_timing)
for features,trace in [(train_features,train_trace),(eval_features,eval_trace)]:
    assert features.loc[features[FLAGS[1]].eq(0),FEATURE].isna().all()
    assert features[FLAGS].isin([0.,1.]).all().all()
    valid=trace.source_available_at.notna()
    assert (pd.to_datetime(trace.loc[valid,'source_available_at'],utc=True)<=pd.to_datetime(trace.loc[valid,'decision_at'],utc=True)).all()
def coverage(x):
    return {'rows':len(x),'seen':int(x[FLAGS[0]].sum()),'comparable':int(x[FLAGS[1]].sum()),
            'known_preannounced':int(x[FLAGS[2]].sum()),'finite_numeric':int(x[FEATURE].notna().sum())}
result={'status':'conditional_input_coverage_verified_not_PIT','plan_sha256':hashlib.sha256((card/'source_preflight_plan.json').read_bytes()).hexdigest(),
        'source_recorder_ids':{key:value['recorder_id'] for key,value in summaries.items()},
        'training_calendar':coverage(train_features),'evaluation_calendar':coverage(eval_features),
        'train_first_decision':str(train_timing.min()),'train_last_decision':str(train_timing.max()),
        'evaluation_first_decision':str(eval_timing.min()),'evaluation_last_decision':str(eval_timing.max()),
        'source_realized_report_events':len(events),'historical_first_public_timestamp_verified':False,
        'all_updates_coverage_verified':False,'original_html_bytes_verified':False,
        'new_model_fits':0,'new_native_accounts':0,'market_outcome_reads':0,
        'interpretation':'Coverage and delayed alignment permit only a historical development design. Exact source/publication and future economic gates remain open.'}
(card/'source_preflight_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False,indent=2))

## 中文结论

此预检只证明来源、保守09:29时点和原Stage1日历的输入覆盖；不会把已知报告状态解释成新增收益。历史首次公开时点、更新完整性和原HTML原字节均未认证。下一步固定模型和完整账户计划、审查实现，再执行一次0bp匹配B/C；失败即关闭该接收层版本。